In [3]:
import numpy as np
import pygame

# 1 = Wall, 0 = Path
# A 20x20 symmetric grid
LAYOUT = [
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
    [1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1],
    [1,0,1,1,0,1,1,1,0,1,1,0,1,1,1,0,1,1,0,1],
    [1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1],
    [1,0,0,0,0,1,1,0,1,1,1,1,0,1,1,0,0,0,0,1],
    [1,0,1,1,0,1,1,0,0,0,0,0,0,1,1,0,1,1,0,1],
    [1,0,0,0,0,0,0,0,1,1,1,1,0,0,0,0,0,0,0,1],
    [1,1,1,0,1,1,1,0,1,0,0,1,0,1,1,1,0,1,1,1],
    [1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1],
    [1,0,1,1,1,1,1,0,0,0,0,0,0,1,1,1,1,1,0,1],
    [1,0,1,1,1,1,1,0,0,0,0,0,0,1,1,1,1,1,0,1],
    [1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1],
    [1,1,1,0,1,1,1,0,1,1,1,1,0,1,1,1,0,1,1,1],
    [1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,0,1,1,0,1,1,0,1,1,1,1,0,1,1,0,1,1,0,1],
    [1,0,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,0,0,1],
    [1,0,1,1,0,0,0,0,1,1,1,1,0,0,0,0,1,1,0,1],
    [1,0,1,1,0,1,1,0,0,1,1,0,0,1,1,0,1,1,0,1],
    [1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
]


pygame 2.6.1 (SDL 2.28.4, Python 3.11.9)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [4]:
def get_state_tensor(pac_pos,ghost_pos,curr_layout):
    state=np.zeros((3,20,20),dtype=np.float32)
    #0 layer is board,1 layer is pacman, 2 layer is ghost
    state[0]=np.array(curr_layout)
    state[1, pac_pos[0], pac_pos[1]] = 1.0
    
    
    state[2, ghost_pos[0], ghost_pos[1]] = 1.0

    return state


In [22]:
class PacMan:
    def __init__(self,board):
        self.board=np.array(board)
        self.reset()
        self.moves = {0: [-1, 0], 1: [1, 0], 2: [0, -1], 3: [0, 1]}


    def reset(self):
        self.pacman_pos=[1,1]
        self.ghost_pos=[18,18]
        self.pellets= (self.board==0).copy()
        return self.get_state()

    def get_state(self):
        state=np.zeros((3,20,20),dtype=np.float32)
        #0 layer is board,1 layer is pacman, 2 layer is ghost
        state[0]=np.array(self.board)
        state[1, self.pacman_pos[0], self.pacman_pos[1]] = 1.0
        
        
        state[2, self.ghost_pos[0], self.ghost_pos[1]] = 1.0

        return state
    
    def step_pac(self,action):
        dr,dc=self.moves[action]
        new_r = self.pacman_pos[0] + dr
        new_c = self.pacman_pos[1] + dc

        reward= -0.1

        done= False

        if self.board[new_r][new_c]==0:
            self.pacman_pos=[new_r,new_c]
            if self.pellets[new_r][new_c]:
                self.pellets[new_r][new_c] = False
                reward+=10
        
        if self.pacman_pos==self.ghost_pos:
            reward+= -200
            done = True
        return self.get_state(),reward,done
    
    def step(self, p_action, g_action):
   
        p_reward = -0.1  
        g_reward = -0.1 
        done = False

      
        p_dr, p_dc = self.moves[p_action]
        p_nr, p_nc = self.pacman_pos[0] + p_dr, self.pacman_pos[1] + p_dc
        if self.board[p_nr][p_nc] == 0:
            self.pacman_pos = [p_nr, p_nc]
            if self.pellets[p_nr][p_nc]:
                self.pellets[p_nr][p_nc] = False
                p_reward += 10

       
        g_dr, g_dc = self.moves[g_action]
        g_nr, g_nc = self.ghost_pos[0] + g_dr, self.ghost_pos[1] + g_dc
        if self.board[g_nr][g_nc] == 0:
            self.ghost_pos = [g_nr, g_nc]

       
        if self.pacman_pos == self.ghost_pos:
            p_reward -= 200  # PACMAN PENALTY
            g_reward += 200  # GHOST REWARD
            done = True
        
        # 4. Check if all pellets are gone (Pacman wins)
        if not np.any(self.pellets):
            p_reward += 500
            done = True

        return self.get_state(), p_reward, g_reward, done
    
    


In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# class PacmanBrain(nn.Module):
#     def __init__(self):
#         super(PacmanBrain, self).__init__()
#         # Input: 3 layers (Board, Pacman, Ghost) of 20x20
#         # Layer 1: Finds small patterns (edges, dots)
#         self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
#         # Layer 2: Finds bigger patterns (corridors, distances)
#         self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        
#         # Flattening 32 channels of 20x20 into one long vector
#         self.fc1 = nn.Linear(32 * 20 * 20, 128)
#         self.fc2 = nn.Linear(128, 4) # Output: The 4 possible moves

#     def forward(self, x):
#         # x is the 20x20 grid
#         x = F.relu(self.conv1(x))
#         x = F.relu(self.conv2(x))
#         x = x.view(x.size(0), -1) # Flatten for the decision layers
#         x = F.relu(self.fc1(x))
#         return self.fc2(x) # These are the "Q-Values" (the guesses)

In [25]:

class PacmanBrain(nn.Module):
    def __init__(self):
        super(PacmanBrain, self).__init__()
        # Input: 3 layers (Board, Pacman, Ghost) of 20x20
        # Layer 1: Finds small patterns (edges, dots)
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
        # Layer 2: Finds bigger patterns (corridors, distances)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        
        # Flattening 32 channels of 20x20 into one long vector
        self.fc1 = nn.Linear(32 * 20 * 20, 128)
        self.fc2 = nn.Linear(128, 4) # Output: The 4 possible moves

    def forward(self, x):
        # x is the 20x20 grid
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1) # Flatten for the decision layers
        x = F.relu(self.fc1(x))
        return self.fc2(x) # These are the "Q-Values" (the guesses)

In [26]:
import random
from collections import deque

class ReplayMemory:
    def __init__(self, capacity=10000):
        self.memory = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

In [27]:
import torch
import torch.optim as optim
import numpy as np
# from models import PacmanBrain, ReplayMemory # From previous steps


env = PacMan(LAYOUT) # Your class
p_brain = PacmanBrain()
g_brain = PacmanBrain()

# Optimizers: These are the "graders" that update the brains
p_optimizer = optim.Adam(p_brain.parameters(), lr=0.001)
g_optimizer = optim.Adam(g_brain.parameters(), lr=0.001)

p_memory = ReplayMemory(10000)
g_memory = ReplayMemory(10000)

epsilon = 1.0  # Start fully random
epsilon_decay = 0.995
batch_size = 32

def train_step(brain, optimizer, memory):
    if len(memory.memory) < batch_size: return
    
    # Grab 32 random memories
    batch = memory.sample(batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)
    
    # Convert to Tensors for PyTorch
    states = torch.FloatTensor(np.array(states))
    actions = torch.LongTensor(actions)
    rewards = torch.FloatTensor(rewards)
    next_states = torch.FloatTensor(np.array(next_states))
    
    # The "Bellman Equation" Core:
    current_q = brain(states).gather(1, actions.unsqueeze(1))
    max_next_q = brain(next_states).detach().max(1)[0]
    expected_q = rewards + (0.99 * max_next_q * (1 - torch.tensor(dones).float()))
    
    # Update the brain weights
    loss = torch.nn.functional.mse_loss(current_q.squeeze(), expected_q)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 2. The Main Loop
for episode in range(500):
    state = env.reset()
    total_p_reward = 0
    done = False
    
    while not done:
        # Choose Actions (Epsilon-Greedy)
        if np.random.rand() < epsilon:
            p_action = np.random.randint(0, 4)
            g_action = np.random.randint(0, 4)
        else:
            with torch.no_grad():
                # Brain makes a guess
                p_action = p_brain(torch.FloatTensor(state).unsqueeze(0)).argmax().item()
                g_action = g_brain(torch.FloatTensor(state).unsqueeze(0)).argmax().item()

        # Step the environment
        next_state, p_reward, g_reward, done = env.step(p_action, g_action)
        
        # Store in memory
        p_memory.push(state, p_action, p_reward, next_state, done)
        g_memory.push(state, g_action, g_reward, next_state, done)
        
        # Learn
        train_step(p_brain, p_optimizer, p_memory)
        train_step(g_brain, g_optimizer, g_memory)
        
        state = next_state
        total_p_reward += p_reward
        
    epsilon = max(0.05, epsilon * epsilon_decay)
    print(f"Episode {episode}: Total Pacman Reward: {total_p_reward}")

Episode 0: Total Pacman Reward: 624.0999999999774
Episode 1: Total Pacman Reward: 45.20000000000064
Episode 2: Total Pacman Reward: 547.2999999999813
Episode 3: Total Pacman Reward: 566.1999999999838
Episode 4: Total Pacman Reward: 593.6999999999944
Episode 5: Total Pacman Reward: 276.2999999999972
Episode 6: Total Pacman Reward: 389.89999999999
Episode 7: Total Pacman Reward: 511.2999999999887
Episode 8: Total Pacman Reward: 410.0999999999781
Episode 9: Total Pacman Reward: 1437.5000000001723
Episode 10: Total Pacman Reward: 471.1999999999896
Episode 11: Total Pacman Reward: 340.9999999999959
Episode 12: Total Pacman Reward: 634.3999999999801
Episode 13: Total Pacman Reward: 369.59999999999513
Episode 14: Total Pacman Reward: 704.9999999999836
Episode 15: Total Pacman Reward: 945.300000000157
Episode 16: Total Pacman Reward: 456.199999999988


KeyboardInterrupt: 

In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

False


AssertionError: Torch not compiled with CUDA enabled